# Procesamiento de datos sobre canciones

- Inteligencia Artificial - grupo 02
- Tarea 3, parte 2. 
- Anthony Sánchez, Andrés Camacho, Enrique Ramírez
- II-2026



In [44]:
import re
import unicodedata
from collections import Counter
import pandas as pd
import math


In [45]:

def preprocesar(texto):
    # pasar a minuscula
    texto = str(texto).lower()
    # quitar tildes
    texto = unicodedata.normalize('NFKD', texto) # esto y la linea que sigue se lo pregunté a claude
    texto = texto.encode('ascii', 'ignore').decode('ascii')
    # retorna terminos a-z
    return re.findall(r'[a-z]+', texto)

def bolsa_de_palabras(texto):
    return Counter(preprocesar(texto)) #esto es un diccionario que cuenta cuantas veces sale una palabra

In [46]:
# Leer hojas de entrenamiento y prueba
train = pd.read_excel('reggaeton_baladas.xlsx', sheet_name='Entrenamiento')
test  = pd.read_excel('reggaeton_baladas.xlsx', sheet_name='Testing')
# Ponerle nombre a columnas
train.columns = ['categoria', 'letra']
test.columns = ['categoria', 'letra']

#Conseguir tokens
def tokenizar(df):
    resultados = []
    for letra in df['letra']:
        resultados.append(preprocesar(letra))
    df['tokens'] = resultados

tokenizar(train)
tokenizar(test)


#visualizar resultado
print('Tamaño de train:', train.shape)  
print('Tamaño de test:', test.shape)
print(train.head())     




Tamaño de train: (552, 3)
Tamaño de test: (138, 3)
   categoria                                              letra  \
0     BALADA  Vuelve a mí\nQue solo y triste me quedé sin ti...   
1     BALADA  Si aún no me lo crees, amor Y quieres, tú, cor...   
2     BALADA  Por cada beso tuyo\nUn sufrimiento recibí sin ...   
3  REGAETTON  Mi bicho anda fugao' y yo quiero que tú me lo ...   
4  REGAETTON  Aquí pasamos moñas por el TSA\nLas putas se mo...   

                                              tokens  
0  [vuelve, a, mi, que, solo, y, triste, me, qued...  
1  [si, aun, no, me, lo, crees, amor, y, quieres,...  
2  [por, cada, beso, tuyo, un, sufrimiento, recib...  
3  [mi, bicho, anda, fugao, y, yo, quiero, que, t...  
4  [aqui, pasamos, monas, por, el, tsa, las, puta...  


In [47]:
# Vocabulario con datos de entrenamiento
# uso set porque no guarda repetidos
def crear_vocabulario(lista_tokens):
    vocabulario = set()
    for tokens in lista_tokens:
        vocabulario.update(tokens)
    return sorted(vocabulario)


vocabulario = crear_vocabulario(train['tokens'])

V = len(vocabulario)

print('Tamaño del vocabulario:', V)
print(vocabulario[:20])


Tamaño del vocabulario: 4158
['a', 'aa', 'ababache', 'abajo', 'abandone', 'abatir', 'abayarde', 'abecedario', 'abierta', 'abiertas', 'abismo', 'abordo', 'abrace', 'abraza', 'abrazado', 'abrazame', 'abrazan', 'abrazando', 'abrazandote', 'abrazare']


In [48]:
# entrenar modelo
def entrenar(df):
    vocabulario = set(crear_vocabulario(df['tokens']))
    V = len(vocabulario)

    clases = ['BALADA', 'REGAETTON']
    priors = {}
    conteos = {}
    total_palabras = {}

    # se filtran las canciones para las dos clases
    for c in clases:
        canciones_c = df[df['categoria'] == c]

        # P(c) = canciones en c / total canciones
        priors[c] = len(canciones_c) / len(df)

        # cuántas veces aparece una palabra en c
        conteos[c] = Counter()
        for tokens in canciones_c['tokens']:
            conteos[c].update(tokens)

        # total de palabras en una clase c
        total_palabras[c] = sum(conteos[c].values())

    # Este return lo hice con claude
    return {'clases': clases, 'vocabulario': vocabulario, 'V': V,
            'priors': priors, 'conteos': conteos, 'total_palabras': total_palabras}


In [49]:
# P(w|c), se usa Laplace
def prob_palabra(modelo, palabra, c):
    conteo = modelo['conteos'][c][palabra]   # por ejemplo, modelo['conteos']['BALADA']['amor']
                                              #  es la cantidad de veces que aparece "amor" en baladas
    return (conteo + 1) / (modelo['total_palabras'][c] + modelo['V'])
# por ejemplo modelo['total_palabras']['BALADA'] es la cantidad de palabras en todas las baladas, incluidas repeticiones


In [50]:
def clasificar(modelo, tokens):
    puntajes = {}
    for c in modelo['clases']:
        # log del prior
        puntaje = math.log(modelo['priors'][c])

        # sumar log de P(w|c) de cada palabra de una cancion
        for w in tokens:
            if w in modelo['vocabulario']: #ingorar palabras que no se conocen
                puntaje += math.log(prob_palabra(modelo, w, c))

        puntajes[c] = puntaje

    # la clase con el puntaje más alto
    prediccion = max(puntajes, key=puntajes.get)
    return prediccion, puntajes


In [51]:
modelo = entrenar(train)

# probar con tres primeras canciones de conjunto de entrenamiento
for i in range(3):
    pred, punt = clasificar(modelo, train['tokens'][i])
    print('Real:', train['categoria'][i], '| Predicción:', pred, punt)

# frases de inventatas
print(clasificar(modelo, preprocesar('te amo mi amor, mi corazón es tuyo')))
print(clasificar(modelo, preprocesar('dale mami perreo hasta abajo')))


Real: BALADA | Predicción: BALADA {'BALADA': -239.98368629708105, 'REGAETTON': -272.0533218817886}
Real: BALADA | Predicción: BALADA {'BALADA': -233.51553648631537, 'REGAETTON': -246.5542862043641}
Real: BALADA | Predicción: BALADA {'BALADA': -234.43493882230555, 'REGAETTON': -255.42957380203765}
('BALADA', {'BALADA': -43.599100657595564, 'REGAETTON': -50.62792887056353})
('REGAETTON', {'BALADA': -45.90416189311212, 'REGAETTON': -34.350585614685144})


In [52]:
def calcular_metricas(reales, predichas, clases):
    # Accuracy
    aciertos = 0
    for r, p in zip(reales, predichas):
        if r == p:
            aciertos += 1
    accuracy = aciertos / len(reales)

    # Precision, recall y F1 por clase. Este for se lo pedí a claude
    precisiones = []
    recalls = []
    f1s = []
    for c in clases:
        vp = 0
        fp = 0
        fn = 0
        for r, p in zip(reales, predichas):
            if p == c and r == c:
                vp += 1
            elif p == c and r != c:
                fp += 1
            elif p != c and r == c:
                fn += 1

        precision = vp / (vp + fp) if vp + fp > 0 else 0
        recall = vp / (vp + fn) if vp + fn > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

        precisiones.append(precision)
        recalls.append(recall)
        f1s.append(f1)

    # Promedio macro
    return {'accuracy': accuracy,
            'precision': sum(precisiones) / len(clases),
            'recall': sum(recalls) / len(clases),
            'f1': sum(f1s) / len(clases)}


In [53]:
# Revolver las canciones
datos = train.sample(frac=1, random_state=67).reset_index(drop=True) # Le pedí a claude que me lo revolviera

k = 5
n = len(datos)
tam = n // k

resultados_cv = []
for i in range(k):
    # límites del fold
    inicio = i * tam
    if i < k - 1:
        fin = (i + 1) * tam
    else:
        fin = n 

    # Separar entrenamiento y validacion
    validacion = datos.iloc[inicio:fin]
    entrenamiento = pd.concat([datos.iloc[:inicio], datos.iloc[fin:]])

    # entrenar modelo
    modelo_cv = entrenar(entrenamiento)

    # predecir canciones validación
    predichas = []
    for tokens in validacion['tokens']:
        pred, _ = clasificar(modelo_cv, tokens)
        predichas.append(pred)

    metricas = calcular_metricas(list(validacion['categoria']), predichas, modelo_cv['clases'])
    resultados_cv.append(metricas)
    print(f'Fold {i+1}: entrenamiento={len(entrenamiento)}, validación={len(validacion)}, {metricas}')

# Promedio de los 5 folds
print('\nPromedio 5-fold CV:')
for m in ['accuracy', 'precision', 'recall', 'f1']:
    valores = [r[m] for r in resultados_cv]
    print(f'{m}: {sum(valores) / k:.4f}')


Fold 1: entrenamiento=442, validación=110, {'accuracy': 0.8909090909090909, 'precision': 0.8961693548387097, 'recall': 0.8898809523809523, 'f1': 0.8903290129611167}
Fold 2: entrenamiento=442, validación=110, {'accuracy': 0.8545454545454545, 'precision': 0.873249299719888, 'recall': 0.8525132275132274, 'f1': 0.8521505376344086}
Fold 3: entrenamiento=442, validación=110, {'accuracy': 0.9090909090909091, 'precision': 0.9160254265640682, 'recall': 0.9116186693147964, 'f1': 0.9089705395564383}
Fold 4: entrenamiento=442, validación=110, {'accuracy': 0.9, 'precision': 0.9089381720430108, 'recall': 0.9028467394902351, 'f1': 0.8997929606625259}
Fold 5: entrenamiento=440, validación=112, {'accuracy': 0.875, 'precision': 0.8952380952380952, 'recall': 0.8710089399744572, 'f1': 0.8723958333333335}

Promedio 5-fold CV:
accuracy: 0.8859
precision: 0.8979
recall: 0.8856
f1: 0.8847


In [54]:
#parte 4
modelo_final = entrenar(train)

# Predecir canciones del conjunto de pruebas
predichas_test = []
for tokens in test['tokens']:
    pred, _ = clasificar(modelo_final, tokens)
    predichas_test.append(pred)

# Métricas
metricas_test = calcular_metricas(list(test['categoria']), predichas_test, modelo_final['clases'])
print('Resultados en Testing:')
for m in ['accuracy', 'precision', 'recall', 'f1']:
    print(f'{m}: {metricas_test[m]:.4f}')

# Mat. de confusión
print('\nMatriz de confusión:')
print(pd.crosstab(test['categoria'], pd.Series(predichas_test, name='prediccion'),
                  rownames=['real'], colnames=['prediccion']))


Resultados en Testing:
accuracy: 0.8333
precision: 0.8582
recall: 0.8235
f1: 0.8267

Matriz de confusión:
prediccion  BALADA  REGAETTON
real                         
BALADA          71          3
REGAETTON       20         44
